# 9. Modelo base: línea base trivial y regresión logística

**Qué hacemos** (ruta A del enunciado):

- **Línea base:** `DummyClassifier` (siempre responde la clase mayoritaria) y **regresión logística** dentro del Pipeline del capítulo 8.
- **Métricas:** accuracy, precision, recall, F1, AUC, Average Precision, matriz de confusión, curvas ROC y PR, calibración.
- **Incertidumbre:** intervalos bootstrap que remuestrean **pacientes**, y diferencia frente a la línea base con los mismos pacientes.
- **Diagnóstico:** curva de aprendizaje, interpretación de coeficientes (odds ratios) y desempeño por subgrupos.
- **Auditoría** del criterio «accuracy ≥ 80–90 %» del enunciado.

La validación cruzada usa solo el entrenamiento (5 pliegues agrupados por paciente); la **prueba se usa una sola vez**, al final.

In [ ]:
from pathlib import Path
import re, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display

project_root = Path.cwd().resolve()
while not (project_root / "_config.yml").exists():
    if project_root.parent == project_root:
        raise FileNotFoundError("No se encontró _config.yml. Ejecuta el notebook dentro de la carpeta del libro.")
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

for _folder in ("tables", "figures"):          # se crean si no existen
    (project_root / _folder).mkdir(exist_ok=True)

from src.project_utils import *

configure_plots()
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 170)
pd.set_option("display.max_colwidth", 70)
TABLES = project_root / "tables"

def md_out(text):
    display(Markdown(text))

# El OneHotEncoder usa handle_unknown="infrequent_if_exist": una categoría que aparece en validación/prueba y no en el ajuste
# se codifica como ceros (comportamiento deseado y documentado). Se silencia solo ese aviso, también en los procesos de joblib.
import os
_msg_onehot = "Found unknown categories"
warnings.filterwarnings("ignore", message=_msg_onehot, category=UserWarning)
os.environ["PYTHONWARNINGS"] = ",".join(filter(None, [os.environ.get("PYTHONWARNINGS", ""), f"ignore:{_msg_onehot}:UserWarning"]))

from sklearn.calibration import calibration_curve
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (ConfusionMatrixDisplay, PrecisionRecallDisplay, RocCurveDisplay, brier_score_loss, f1_score,
                             make_scorer, precision_recall_curve, precision_score, recall_score, roc_curve)
from sklearn.model_selection import cross_val_predict, cross_validate, learning_curve

pop = pd.read_csv(TABLES / "decision_poblacion.csv").iloc[0]
excluir = bool(pop["excluir_terminales"])
dec = pd.read_csv(TABLES / "decision_transformacion.csv").iloc[0]
cm = pd.read_csv(TABLES / "columnas_modelo.csv")
num_lin, num_log, cat = [cm.loc[cm["grupo"] == g, "variable"].tolist() for g in ("num_lin", "num_log", "cat")]
todas = num_lin + num_log + cat

train, test = load_train(include_terminal=not excluir), load_test(include_terminal=not excluir)
Ftr, Fte = prepare_frame(train), prepare_frame(test)
Xtr, ytr, gtr = Ftr[todas], Ftr[TARGET], Ftr[GROUP_COL]
Xte, yte, gte = Fte[todas], Fte[TARGET], Fte[GROUP_COL]
folds = cv_folds(ytr, gtr)
def nuevo_modelo():
    return make_model_pipeline(num_lin, num_log, cat)

In [ ]:
md_out(f"- **Entrenamiento:** {es(len(train), 0)} encuentros de {es(gtr.nunique(), 0)} pacientes; reingreso <30 {es(ytr.mean(), 1, pct=True)}\n"
       f"- **Prueba:** {es(len(test), 0)} encuentros de {es(gte.nunique(), 0)} pacientes; reingreso <30 {es(yte.mean(), 1, pct=True)}\n"
       f"- **Variables del modelo:** {len(todas)} ({len(num_lin) + len(num_log)} numéricas, {len(cat)} categóricas); "
       f"`log1p` en los conteos: {'sí' if bool(dec['usar_log1p']) else 'no'} (decisión del capítulo 8)")

## 1. Validación cruzada en entrenamiento

Cinco pliegues estratificados y agrupados por paciente. Se reporta media ± desviación entre pliegues. El Dummy no aprende nada:
sirve para mostrar por qué un accuracy alto puede coexistir con recall 0 para la clase `<30`.

In [ ]:
scoring = {"accuracy": "accuracy", "precision": make_scorer(precision_score, zero_division=0),
           "recall": make_scorer(recall_score, zero_division=0), "f1": make_scorer(f1_score, zero_division=0),
           "roc_auc": "roc_auc", "average_precision": "average_precision"}
modelos = {"Dummy (clase mayoritaria)": DummyClassifier(strategy="most_frequent"), "Regresión logística": nuevo_modelo()}
filas = []
for nombre, m in modelos.items():
    r = cross_validate(m, Xtr, ytr, cv=folds, scoring=scoring, n_jobs=2, error_score="raise")
    for k in scoring:
        filas.append({"modelo": nombre, "métrica": k, "media": r[f"test_{k}"].mean(), "desv_est": r[f"test_{k}"].std()})
cv_res = pd.DataFrame(filas)
cv_res.to_csv(TABLES / "cv_resumen.csv", index=False)
tabla_cv = cv_res.pivot(index="métrica", columns="modelo", values="media").loc[list(scoring)]
tabla_sd = cv_res.pivot(index="métrica", columns="modelo", values="desv_est").loc[list(scoring)]
display(tabla_cv.round(4).astype(str) + " ± " + tabla_sd.round(4).astype(str))
lr_cv, d_cv = tabla_cv["Regresión logística"], tabla_cv["Dummy (clase mayoritaria)"]
md_out(f"En validación cruzada la regresión logística obtiene un AUC de **{es(lr_cv['roc_auc'], 3)}** frente a 0,5 del Dummy, y un accuracy de "
       f"**{es(lr_cv['accuracy'], 3)}** frente a {es(d_cv['accuracy'], 3)}. La mejora de accuracy es de **{es(100 * (lr_cv['accuracy'] - d_cv['accuracy']), 1)} puntos porcentuales**, pero la comparación principal debe considerar recall, F1 y Average Precision por el desbalance.")

## 2. Ajuste final y evaluación en la prueba

Los dos modelos se ajustan con todo el entrenamiento y se evalúan **una sola vez** en la prueba, con umbral 0,5.

In [ ]:
dummy = DummyClassifier(strategy="most_frequent").fit(Xtr, ytr)
lr = nuevo_modelo().fit(Xtr, ytr)
p_dummy, p_lr = dummy.predict_proba(Xte)[:, 1], lr.predict_proba(Xte)[:, 1]
pred_dummy, pred_lr = dummy.predict(Xte), (p_lr >= 0.5).astype(int)

res = pd.DataFrame({"Dummy (clase mayoritaria)": classification_metrics(yte, pred_dummy, p_dummy),
                    "Regresión logística": classification_metrics(yte, pred_lr, p_lr)}).T
res["Brier"] = [brier_score_loss(yte, p_dummy), brier_score_loss(yte, p_lr)]
res.to_csv(TABLES / "resultados_holdout.csv")
display(res.round(4))
md_out(f"En la prueba: accuracy **{es(res.loc['Regresión logística', 'Accuracy'], 3)}** (Dummy {es(res.loc['Dummy (clase mayoritaria)', 'Accuracy'], 3)}), "
       f"AUC **{es(res.loc['Regresión logística', 'ROC-AUC'], 3)}**, recall {es(res.loc['Regresión logística', 'Recall'], 3)}, "
       f"precision {es(res.loc['Regresión logística', 'Precision'], 3)}, F1 {es(res.loc['Regresión logística', 'F1'], 3)}.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, nombre, pred in zip(axes, ["Dummy (clase mayoritaria)", "Regresión logística"], [pred_dummy, pred_lr]):
    ConfusionMatrixDisplay.from_predictions(yte, pred, ax=ax, display_labels=["no <30", "reingreso <30"], colorbar=False, cmap="Blues")
    ax.set_title(nombre)
plt.tight_layout(); save_figure(fig, "09_matrices_confusion.png"); plt.show()
tn, fp, fn, tp = skm.confusion_matrix(yte, pred_lr).ravel()
md_out(f"TN={es(tn, 0)}, FP={es(fp, 0)}, FN={es(fn, 0)} y TP={es(tp, 0)}. Un **falso negativo** es un encuentro que sí terminó en reingreso antes de 30 días pero fue clasificado como clase 0; si el modelo se usara para priorización, ese caso podría no recibir seguimiento reforzado. Esta es una interpretación operativa, no una afirmación causal o médica individual.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
RocCurveDisplay.from_predictions(yte, p_lr, name="Regresión logística", ax=axes[0], color="#e45756")
axes[0].plot([0, 1], [0, 1], "--", color="gray", label="Dummy / azar"); axes[0].legend(); axes[0].set_title("Curva ROC (prueba)")
PrecisionRecallDisplay.from_predictions(yte, p_lr, name="Regresión logística", ax=axes[1], color="#e45756")
axes[1].axhline(yte.mean(), ls="--", color="gray", label=f"Dummy (prevalencia {yte.mean():.2f})"); axes[1].legend(); axes[1].set_title("Curva Precision-Recall (prueba)")
plt.tight_layout(); save_figure(fig, "09_curvas_roc_pr.png"); plt.show()

## 3. Calibración

Una probabilidad está **calibrada** si, entre los pacientes a los que el modelo les asigna 30 %, reingresa cerca del 30 %. Se revisa en la prueba y,
con predicciones **fuera de pliegue**, en el entrenamiento. El **Brier** mide el error cuadrático de las probabilidades (menor es mejor); se compara
con el de predecir siempre la prevalencia.

In [ ]:
oof = cross_val_predict(nuevo_modelo(), Xtr, ytr, cv=folds, method="predict_proba", n_jobs=2)[:, 1]
f_te, m_te = calibration_curve(yte, p_lr, n_bins=10, strategy="quantile")
f_oof, m_oof = calibration_curve(ytr, oof, n_bins=10, strategy="quantile")
fig, ax = plt.subplots(figsize=(5.5, 5))
ax.plot([0, 1], [0, 1], "--", color="gray", label="calibración perfecta")
ax.plot(m_te, f_te, "o-", color="#e45756", label="prueba"); ax.plot(m_oof, f_oof, "s-", color="#4c78a8", label="entrenamiento (fuera de pliegue)")
ax.set_xlabel("probabilidad predicha"); ax.set_ylabel("fracción real de reingresos"); ax.legend(); ax.set_title("Diagrama de fiabilidad")
plt.tight_layout(); save_figure(fig, "09_calibracion.png"); plt.show()
brier_lr, brier_ref = brier_score_loss(yte, p_lr), brier_score_loss(yte, np.full(len(yte), ytr.mean()))
desv = float(np.mean(np.abs(f_te - m_te)))
md_out(f"Brier de la regresión logística: **{es(brier_lr, 4)}**; de predecir siempre la prevalencia: {es(brier_ref, 4)}. "
       f"La desviación media entre lo predicho y lo observado por tramos es de {es(100 * desv, 1)} puntos porcentuales: "
       + ("las probabilidades están **bien calibradas** (sin ajuste adicional)." if desv < 0.03 else "hay descalibración apreciable; convendría recalibrar antes de usar las probabilidades."))

## 4. Umbral de decisión

El umbral 0,5 no es sagrado. Se elige un **umbral alternativo usando solo las predicciones fuera de pliegue del entrenamiento** (el que maximiza el índice de
Youden: recall − tasa de falsos positivos) y se evalúa en la prueba. Un umbral más bajo detecta más reingresos a costa de más falsas alarmas.

In [ ]:
fpr, tpr, thr = roc_curve(ytr, oof)
umbral = float(thr[np.argmax(tpr - fpr)])
def a_umbral(p, u):
    pred = (p >= u).astype(int)
    fila = classification_metrics(yte, pred, p)
    fila["Balanced accuracy"] = skm.balanced_accuracy_score(yte, pred)
    return fila
comp_u = pd.DataFrame({"umbral 0,5": a_umbral(p_lr, 0.5), f"umbral {umbral:.3f} (Youden, entrenamiento)": a_umbral(p_lr, umbral)}).T
comp_u.to_csv(TABLES / "umbral_alternativo.csv")
display(comp_u.round(4))
prec, rec, th = precision_recall_curve(ytr, oof)
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(th, prec[:-1], label="precision"); ax.plot(th, rec[:-1], label="recall"); ax.axvline(umbral, color="gray", ls="--", label="umbral elegido")
ax.set_xlabel("umbral"); ax.legend(); ax.set_title("Precision y recall según el umbral (entrenamiento, fuera de pliegue)")
plt.tight_layout(); save_figure(fig, "09_umbral.png"); plt.show()

## 5. Intervalos de confianza: bootstrap por paciente

Como un paciente tiene varios encuentros, remuestrear **filas** subestimaría la incertidumbre. Se remuestrean **pacientes completos** (1.000 réplicas) y se
calcula el intervalo percentil del 95 %. Para comparar con el Dummy se usan **los mismos pacientes remuestreados** en ambos modelos.

In [ ]:
ic = cluster_bootstrap(yte, pred_lr, p_lr, gte, B=1000)
ic_d = cluster_bootstrap(yte, pred_dummy, p_dummy, gte, B=1000)
dif = paired_cluster_bootstrap(yte, pred_lr, p_lr, pred_dummy, p_dummy, gte, B=1000)
ic.to_csv(TABLES / "intervalos_bootstrap_logistica.csv")
dif.to_csv(TABLES / "diferencias_vs_dummy.csv")
display(ic.round(4)); display(dif.round(4))

comprobacion_ic = pd.DataFrame({
    "métrica": dif.index,
    "diferencia_logistica_menos_dummy": dif["Diferencia"].values,
    "ic_inferior": dif["IC inferior"].values,
    "ic_superior": dif["IC superior"].values,
    "ic_excluye_cero": ((dif["IC inferior"] > 0) | (dif["IC superior"] < 0)).values,
})
comprobacion_ic.to_csv(TABLES / "comprobacion_ic_por_metrica.csv", index=False)
display(comprobacion_ic.round(4))
for metric in ["Accuracy", "Recall", "F1", "ROC-AUC", "Average Precision"]:
    row = dif.loc[metric]
    conclusion = "excluye cero" if (row["IC inferior"] > 0 or row["IC superior"] < 0) else "incluye cero"
    md_out(f"- **{metric}:** diferencia {es(row['Diferencia'], 3)}, IC 95 % {es(row['IC inferior'], 3)} a {es(row['IC superior'], 3)}; el intervalo **{conclusion}**.")
n_pos_pred = int(tp + fp)
md_out(f"- **Precisión con umbral 0,5:** el valor {es(ic.loc['Precision', 'Estimación'], 3)} se apoya en solo **{n_pos_pred} predicciones positivas** "
       f"({tp} aciertos) entre {es(len(yte), 0)} encuentros de prueba; su IC bootstrap va de {es(ic.loc['Precision', 'IC 95 % inferior'], 2)} "
       f"a {es(ic.loc['Precision', 'IC 95 % superior'], 2)}. **No debe interpretarse como un desempeño estable** del modelo.")

## 6. Curva de aprendizaje

Se utiliza **Average Precision**, más informativa que accuracy bajo desbalance.
Los folds de validación respetan `patient_nbr`; ningún paciente de validación
aparece en el entrenamiento del mismo fold.

In [ ]:
tam, tr_s, va_s = learning_curve(
    nuevo_modelo(), Xtr, ytr, cv=folds,
    train_sizes=[0.1, 0.25, 0.5, 0.75, 1.0],
    scoring="average_precision", n_jobs=2,
)
curva_aprendizaje = pd.DataFrame({
    "n_train": tam,
    "ap_entrenamiento_media": tr_s.mean(axis=1),
    "ap_entrenamiento_desv": tr_s.std(axis=1),
    "ap_validacion_media": va_s.mean(axis=1),
    "ap_validacion_desv": va_s.std(axis=1),
})
curva_aprendizaje.to_csv(TABLES / "curva_aprendizaje.csv", index=False)
display(curva_aprendizaje.round(4))

fig, ax = plt.subplots(figsize=(7, 4.2))
for s, nombre, c in [(tr_s, "entrenamiento", "#4c78a8"), (va_s, "validación", "#e45756")]:
    ax.plot(tam, s.mean(axis=1), "o-", color=c, label=nombre)
    ax.fill_between(tam, s.mean(axis=1) - s.std(axis=1), s.mean(axis=1) + s.std(axis=1), color=c, alpha=0.15)
ax.set_xlabel("encuentros de entrenamiento"); ax.set_ylabel("Average Precision")
ax.legend(); ax.set_title("Curva de aprendizaje (validación agrupada por paciente)")
plt.tight_layout(); save_figure(fig, "09_curva_aprendizaje.png"); plt.show()
brecha = tr_s.mean(axis=1)[-1] - va_s.mean(axis=1)[-1]
ultimo = va_s.mean(axis=1)[-1] - va_s.mean(axis=1)[-2]
md_out(f"Con todos los datos, el Average Precision es {es(tr_s.mean(axis=1)[-1], 3)} en entrenamiento y {es(va_s.mean(axis=1)[-1], 3)} en validación (brecha {es(brecha, 3)}). "
       f"Pasar de 75 % a 100 % de los datos cambia el Average Precision de validación en {es(ultimo, 4)}. "
       + ("La curva **se aplana**: más datos con este mismo modelo aportarían poco; la mejora vendría de mejores variables o de un modelo más flexible." if ultimo < 0.003 else
          "La curva sigue subiendo: más datos podrían ayudar."))

## 7. Interpretación de coeficientes y odds ratios

Las numéricas se interpretan por una desviación estándar. En las categóricas,
`OneHotEncoder(drop='first')` elimina una categoría por variable: cada odds
ratio compara el nivel indicado con esa referencia, manteniendo las demás
variables constantes. Los resultados son asociaciones ajustadas, no efectos
causales.

In [ ]:
orr = odds_ratio_table(lr)
referencias = onehot_reference_table(lr, cat)
orr.to_csv(TABLES / "odds_ratios.csv", index=False)
referencias.to_csv(TABLES / "categorias_referencia_onehot_final.csv", index=False)
display(orr.head(15).round(3)); display(referencias.head(12))
t = orr.head(15).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.barh(t["variable"].str.replace("cat__", "").str.replace("numlog__", "").str.replace("num__", ""), t["odds_ratio"] - 1,
        color=np.where(t["odds_ratio"] > 1, "#e45756", "#4c78a8"))
ax.axvline(0, color="black", lw=1); ax.set_xlabel("odds ratio − 1")
ax.set_title("15 coeficientes de mayor magnitud"); plt.tight_layout(); save_figure(fig, "09_odds_ratios.png"); plt.show()
sube, baja = orr[orr["odds_ratio"] > 1].iloc[0], orr[orr["odds_ratio"] < 1].iloc[0]
md_out(f"La variable transformada con mayor asociación positiva es `{sube['variable']}` (OR {es(sube['odds_ratio'], 2)}) y la de mayor asociación negativa es `{baja['variable']}` (OR {es(baja['odds_ratio'], 2)}). La interpretación exacta de variables categóricas debe consultar su referencia en la tabla anterior. No se atribuye causalidad.")

## 8. Desempeño por subgrupos

Por razones éticas se revisa si el modelo rinde de forma parecida en subgrupos de sexo, raza y edad (solo grupos con al menos 300 encuentros de prueba y
ambas clases). Diferencias grandes indicarían que el modelo puede perjudicar más a un grupo.

In [ ]:
Fte = Fte.assign(grupo_edad=pd.cut(Fte["age_mid"], [0, 50, 70, 200], right=False, labels=["<50", "50–69", "70+"]).astype("object"))
def por_grupo(col, min_n=300, u=0.5):
    s = Fte[col].astype("object").where(Fte[col].notna(), "Missing")
    filas = []
    for val in s.unique():
        m = (s == val).to_numpy()
        if m.sum() < min_n or yte[m].nunique() < 2:
            continue
        pr = (p_lr[m] >= u).astype(int)
        filas.append({col: val, "encuentros": int(m.sum()), "reingreso_%": 100 * yte[m].mean(), "AUC": skm.roc_auc_score(yte[m], p_lr[m]),
                      "recall": skm.recall_score(yte[m], pr, zero_division=0), "precision": skm.precision_score(yte[m], pr, zero_division=0),
                      "accuracy": skm.accuracy_score(yte[m], pr)})
    return pd.DataFrame(filas).sort_values("encuentros", ascending=False)
sub = {c: por_grupo(c) for c in ["gender", "race", "grupo_edad"]}
todos = pd.concat([d.rename(columns={c: "grupo"}).assign(variable=c) for c, d in sub.items()])
todos.to_csv(TABLES / "desempeno_subgrupos.csv", index=False)
for c, d in sub.items():
    display(d.round(3))
rango = todos["AUC"].max() - todos["AUC"].min()
md_out(f"El AUC por subgrupo va de **{es(todos['AUC'].min(), 3)}** a **{es(todos['AUC'].max(), 3)}** (rango {es(rango, 3)}). "
       + ("El rendimiento es **parecido** entre subgrupos." if rango < 0.05 else "Hay diferencias entre subgrupos que deben vigilarse antes de cualquier uso."))

### Subgrupos con el umbral alternativo

Con el umbral 0,5 el modelo casi no predice positivos (recall global muy cercano a 0), así que el recall y la precisión por subgrupo no permiten comparar grupos. Se repite la tabla con el **umbral de Youden elegido en la sección 4 con las predicciones fuera de pliegue del entrenamiento** (no se recalcula con la prueba); los mismos criterios de inclusión: al menos 300 encuentros y ambas clases.

In [ ]:
sub_alt = {c: por_grupo(c, u=umbral) for c in ["gender", "race", "grupo_edad"]}
todos_alt = pd.concat([d.rename(columns={c: "grupo"}).assign(variable=c) for c, d in sub_alt.items()])
todos_alt = todos_alt.assign(umbral=umbral)
todos_alt.to_csv(TABLES / "desempeno_subgrupos_umbral_alt.csv", index=False)
md_out(f"Umbral aplicado: **{es(umbral, 3)}** (Youden, entrenamiento fuera de pliegue).")
for c, d in sub_alt.items():
    display(d.round(3))
def extremos(t, met):
    g = t.columns[0]                      # columna con el nombre del grupo (gender, race o grupo_edad)
    t = t.sort_values(met); lo, hi = t.iloc[0], t.iloc[-1]
    return f"de {es(lo[met], 3)} (`{lo[g]}`) a {es(hi[met], 3)} (`{hi[g]}`)"
pred_alt = (p_lr >= umbral).astype(int)
glob_rec, glob_pre = skm.recall_score(yte, pred_alt), skm.precision_score(yte, pred_alt)
lineas = []
for c, d in sub_alt.items():
    lineas.append(f"- **{c}:** recall {extremos(d, 'recall')}; precisión {extremos(d, 'precision')}.")
grandes = todos_alt[todos_alt["encuentros"] >= 1000]
pequenos = todos_alt[todos_alt["encuentros"] < 1000].assign(positivos=lambda d: (d["encuentros"] * d["reingreso_%"] / 100).round())
r_rec, r_pre = grandes["recall"].max() - grandes["recall"].min(), grandes["precision"].max() - grandes["precision"].min()
md_out(f"Con este umbral el recall global es **{es(glob_rec, 3)}** y la precisión global **{es(glob_pre, 3)}** (frente a un recall prácticamente nulo con 0,5).\n\n"
       + "\n".join(lineas) + "\n\n"
       f"En los grupos con al menos 1.000 encuentros, el recall varía {es(r_rec, 3)} y la precisión solo {es(r_pre, 3)}: con un mismo umbral, "
       f"la **precisión es bastante homogénea** (entre {es(100 * todos_alt['precision'].min(), 0)} % y {es(100 * todos_alt['precision'].max(), 0)} % de las alertas son reingresos reales en todos los grupos), mientras que el **recall "
       "difiere más**, es decir, la proporción de reingresos que se detecta no es igual entre grupos. En edad, el grupo de mayor recall recibe también "
       "más alertas (menor accuracy), por lo que su mayor detección se paga con más falsas alarmas. "
       + (f"Los grupos pequeños ({', '.join(f'`{g}`' for g in pequenos['grupo'])}) tienen como máximo {int(pequenos['positivos'].max())} reingresos cada uno y sus cifras son inestables; no deben leerse como diferencias reales. " if len(pequenos) else "")
       + "Estas diferencias deben vigilarse antes de cualquier uso; el umbral se fijó con el entrenamiento y la prueba solo se usa para describirlas.")

## 9. Auditoría del criterio «accuracy ≥ 80–90 %»

El diagrama del enunciado pide, si el accuracy (o R²) llega a 80–90 %, **verificar fuga, comparar con la línea base, revisar la complejidad y evaluar si el problema es
trivial**. Se responde con los números de este libro.

In [ ]:
acc_lr, acc_d = res.loc["Regresión logística", "Accuracy"], res.loc["Dummy (clase mayoritaria)", "Accuracy"]
auc_lr = res.loc["Regresión logística", "ROC-AUC"]
fuga = pd.read_csv(TABLES / "auditoria_fuga.csv")
# Solo predictoras candidatas: se excluyen el objetivo y su fuente (decision == "Excluir del conjunto X", AUC = 1 por construcción)
# y los identificadores, que nunca entran al modelo.
fuga_x = fuga[(fuga["decision"] != "Excluir del conjunto X") & (~fuga["variable"].isin([ID_COL, GROUP_COL]))]
fila_max = fuga_x.loc[fuga_x["AUC univariado"].idxmax()]
max_uni, var_max = float(fila_max["AUC univariado"]), str(fila_max["variable"])
activa = bool(acc_lr >= 0.80)
audit = pd.DataFrame([
    {"criterio": "Accuracy del modelo en la prueba", "resultado": es(acc_lr, 3), "umbral": "≥ 0,80", "se cumple": acc_lr >= 0.80},
    {"criterio": "Accuracy de la línea base trivial", "resultado": es(acc_d, 3), "umbral": "—", "se cumple": acc_d >= 0.80},
    {"criterio": "Mejora del modelo sobre la línea base (pp)", "resultado": es(100 * (acc_lr - acc_d), 1), "umbral": "—", "se cumple": None},
    {"criterio": "AUC del modelo", "resultado": es(auc_lr, 3), "umbral": "≥ 0,90 sospechoso", "se cumple": auc_lr >= 0.90},
    {"criterio": "Mayor AUC univariado de una predictora candidata (capítulo 7)", "resultado": es(max_uni, 3), "umbral": "> 0,90 alerta de fuga", "se cumple": max_uni > 0.90, "variable": var_max},
    {"criterio": "Variables del modelo tras codificar", "resultado": str(len(lr.named_steps['prep'].get_feature_names_out())), "umbral": "—", "se cumple": None},
])
audit = audit[["criterio", "variable", "resultado", "umbral", "se cumple"]]
audit.to_csv(TABLES / "auditoria_nota_critica.csv", index=False)
display(audit)
md_out("**Lectura de la auditoría:**\n\n"
       f"- **Criterio del diagrama:** el accuracy del modelo es {es(acc_lr, 3)}, "
       + ("**inferior a 0,80: el diagrama conduce directamente al entregable**, sin necesidad de cambiar de dataset." if not activa else
          "**igual o superior a 0,80: se activa la revisión** (fuga, línea base, complejidad).") + "\n"
       f"- **Línea base:** el Dummy obtiene {es(acc_d, 3)}; el modelo mejora {es(100 * (acc_lr - acc_d), 1)} puntos porcentuales.\n"
       f"- **Fuga:** la mayor señal individual entre las predictoras candidatas es {es(max_uni, 3)} (`{var_max}`)"
       + (" (sin variable sospechosa); el objetivo, su fuente `readmitted` y los identificadores se excluyen del cálculo porque no forman parte de X." if max_uni <= 0.90 else " (variable sospechosa: revisar).") + "\n"
       f"- **Complejidad:** un modelo lineal con AUC {es(auc_lr, 3)}"
       + (" no es un problema trivial: hay margen de mejora con modelos más flexibles." if auc_lr < 0.85 else " es alto; conviene comparar con modelos más simples y revisar de nuevo la fuga.") )

## 10. Residuos

En clasificación no hay residuos continuos como en regresión; su papel lo cumplen la **matriz de confusión** (errores por clase), la **calibración**
(errores en las probabilidades) y el **desempeño por subgrupos** (dónde falla el modelo). Se declaran aquí como el análogo del análisis de residuos.